<a href="https://colab.research.google.com/github/ArinzeIhematulam/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArinzeIhematulam/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule, in plain words: "A page is worth reviewing for a title/meta rewrite if it's getting meaningfully fewer clicks than other pages sitting at the same search position — and the more impressions it has, the more that click gap actually costs."

Signal 1 — staleness (flag-linked, behind FlyRank's refresh flags). Verdict: MIXED. Measuring content_updated_date as of March 1 (before the decision window), only 8,042 of 51,586 rows (15.6%) even land in a valid staleness bucket — most content was updated after March 1 relative to this slice, so staleness can't describe most of the population here. Within the valid buckets, the pattern also isn't clean: 90–180d shows a higher decline rate (0.753, n=89) than both <90d (0.627, n=7,948) and 180–365d (0.600, n=5, too small to trust). No reliable "staler = more decline" signal — a real, useful negative that saved me from building a rule on it.

Signal 2 — CTR vs. position (flag-linked, the CTR-fix logic). Verdict: CONFIRMED. Measured over days 1–15 only (the decision moment, no outcome-window overlap): CTR falls cleanly as position worsens across the four largest tiers — top_3 1.45% (n=7,638) → 4-10 1.04% (n=27,237) → 11-20 0.90% (n=9,002) → 21-50 0.79% (n=7,370), covering 99.4% of rows. The 50+ tier reverses (3.11%, n=315, 0.6% of rows) — likely a small, different population, not counter-evidence against the main pattern. This signal is real and became the basis for the rule.

Reason codes: ctr_below_tier_average (the only one this version of the rule produces — a page's h1 CTR sits below the average CTR of other pages at the same position tier).

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in sorted(files):
    print(f)
schema = con.sql("""
    DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' LIMIT 1
""").df()
print(schema.to_string())

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

##Step 1 — build the joined base table.

In [30]:
import pandas as pd

base = con.sql("""
    WITH filtered AS (
        SELECT *, EXTRACT(day FROM report_date) AS day_of_month
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    h1 AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_h1, SUM(gsc_clicks) AS clicks_h1
        FROM filtered WHERE day_of_month <= 15 GROUP BY content_hash_id
    ),
    h2 AS (
        SELECT content_hash_id, SUM(gsc_clicks) AS clicks_h2
        FROM filtered WHERE day_of_month > 15 GROUP BY content_hash_id
    ),
    full_month AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_month,
               SUM(gsc_clicks) AS clicks_month, AVG(gsc_avg_position) AS avg_position_month
        FROM filtered GROUP BY content_hash_id
    ),
    dc AS (
        SELECT content_hash_id, content_updated_date
        FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
    SELECT h1.content_hash_id, h1.impressions_h1, h1.clicks_h1,
           h2.clicks_h2, full_month.impressions_month, full_month.clicks_month, full_month.avg_position_month,
           DATE_DIFF('day', dc.content_updated_date, DATE '2026-03-01') AS days_since_update
    FROM h1
    JOIN h2 USING (content_hash_id)
    JOIN full_month USING (content_hash_id)
    LEFT JOIN dc USING (content_hash_id)
    WHERE h1.clicks_h1 > 0
""").df()

base["declining"] = (base["clicks_h2"] < base["clicks_h1"]).astype(int)
print(f"Rows: {len(base)}")
print(base["days_since_update"].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 51586
count    51586.000000
mean       -86.162932
std         43.283569
min       -127.000000
25%       -120.000000
50%       -102.000000
75%        -80.000000
max        234.000000
Name: days_since_update, dtype: float64


##Step 2 — Signal 1 (staleness, flag-linked)

In [31]:
bins = [-1, 90, 180, 365, 100000]
tier_labels = ["<90d", "90-180d", "180-365d", "365d+"]
base["staleness_tier"] = pd.cut(base["days_since_update"], bins=bins, labels=tier_labels)

staleness_table = base.groupby("staleness_tier", observed=True)["declining"].agg(["mean", "count"])
print(staleness_table)

                    mean  count
staleness_tier                 
<90d            0.627453   7948
90-180d         0.752809     89
180-365d        0.600000      5


##Step 3 — Signal 2 (CTR vs. position, the CTR-fix logic):

In [32]:
base["ctr_month"] = base["clicks_month"] / base["impressions_month"]
pos_bins = [0, 3, 10, 20, 50, 100000]
pos_labels = ["top_3", "4-10", "11-20", "21-50", "50+"]
base["position_tier"] = pd.cut(base["avg_position_month"], bins=pos_bins, labels=pos_labels)

ctr_table = base.groupby("position_tier", observed=True)["ctr_month"].agg(["mean", "count"])
print(ctr_table)

                   mean  count
position_tier                 
top_3          0.008329   6098
4-10           0.007390  28481
11-20          0.006193   9381
21-50          0.004477   7352
50+            0.011246    273


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Add h1-only position, needed for a leakage-clean score
base2 = con.sql("""
    WITH filtered AS (
        SELECT *, EXTRACT(day FROM report_date) AS day_of_month
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND EXTRACT(day FROM report_date) <= 15
    )
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_h1_full,
           SUM(gsc_clicks) AS clicks_h1_full, AVG(gsc_avg_position) AS avg_position_h1
    FROM filtered GROUP BY content_hash_id
""").df()

base = base.merge(base2[["content_hash_id", "avg_position_h1"]], on="content_hash_id", how="left")

# Rebuild position tiers and CTR using ONLY h1 (day 1-15) — no outcome-window overlap
base["ctr_h1"] = base["clicks_h1"] / base["impressions_h1"]
pos_bins = [0, 3, 10, 20, 50, 100000]
pos_labels = ["top_3", "4-10", "11-20", "21-50", "50+"]
base["position_tier_h1"] = pd.cut(base["avg_position_h1"], bins=pos_bins, labels=pos_labels)

tier_avg_ctr_h1 = base.groupby("position_tier_h1", observed=True)["ctr_h1"].transform("mean")
base["ctr_gap_h1"] = tier_avg_ctr_h1 - base["ctr_h1"]

base["score"] = (base["ctr_gap_h1"] * base["impressions_h1"]).clip(lower=0)
base["reason_code"] = "ctr_below_tier_average"
base["action"] = base["score"].apply(lambda s: "review_title_meta" if s > 0 else "monitor")

ranked = base.sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(ranked)} rows to work/outputs/baseline_action_score.csv")

print(f"\nBase rate (declining): {ranked['declining'].mean():.3f}")
for k in [10, 20, 50]:
    p = ranked.head(k)["declining"].mean()
    print(f"Precision@{k}: {p:.3f}")

ranked[["content_hash_id","impressions_h1","ctr_h1","position_tier_h1","ctr_gap_h1","score","reason_code","action"]].head(10)

Wrote 51586 rows to work/outputs/baseline_action_score.csv

Base rate (declining): 0.556
Precision@10: 0.700
Precision@20: 0.650
Precision@50: 0.600


,content_hash_id,impressions_h1,ctr_h1,position_tier_h1,ctr_gap_h1,score,reason_code,action
0,content_ec2e0346994fb5a5,132811.0,0.006347,top_3,0.008108,1076.832632,ctr_below_tier_average,review_title_meta
1,content_34a70fea29d15f24,73639.0,0.000244,top_3,0.014211,1046.479261,ctr_below_tier_average,review_title_meta
2,content_e8a52cf3d5988c07,143173.0,0.002466,11-20,0.006513,932.550389,ctr_below_tier_average,review_title_meta
3,content_6aa54d6bbdbf6f24,27832.0,0.000072,50+,0.031039,863.866350,ctr_below_tier_average,review_title_meta
4,content_7c6373141eae744a,86860.0,0.000587,4-10,0.009809,851.998345,ctr_below_tier_average,review_title_meta
5,content_b99ea6861864dea5,91474.0,0.002022,4-10,0.008374,765.965584,ctr_below_tier_average,review_title_meta
6,content_36e53e9c707674fc,109909.0,0.001046,21-50,0.006866,754.591078,ctr_below_tier_average,review_title_meta
7,content_acbcc847f8996314,83715.0,0.001589,4-10,0.008807,737.302860,ctr_below_tier_average,review_title_meta
8,content_2de9a39d3482a269,21302.0,0.000141,50+,0.030970,659.715040,ctr_below_tier_average,review_title_meta
9,content_7172a7fad43f0998,108663.0,0.004537,4-10,0.005859,636.662781,ctr_below_tier_average,review_title_meta


In [34]:
ctr_table_h1 = base.groupby("position_tier_h1", observed=True)["ctr_h1"].agg(["mean", "count"])
print(ctr_table_h1)

                      mean  count
position_tier_h1                 
top_3             0.014455   7638
4-10              0.010396  27237
11-20             0.008979   9002
21-50             0.007912   7370
50+               0.031110    315


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*


1. content_ec2e0346994fb5a5 — action: review_title_meta. Why: top_3 position but only 0.63% CTR vs. 1.45% tier average, on 132,811 impressions. What would make it wrong: if this page's real intent is informational/no-click-needed (e.g. answers the query directly in the snippet), low CTR could be by design, not a fixable weakness.

2. content_34a70fea29d15f24 — review_title_meta. Why: top_3 position, near-zero CTR (0.02%) despite 73,639 impressions — the single biggest ctr_gap in the set. Wrong if: a technical issue (broken link, wrong canonical) is misreported as a title/meta problem — worth a manual click-through check first.

3. content_e8a52cf3d5988c07 — review_title_meta. Why: 11-20 tier, CTR half the tier average on 143k impressions. Wrong if: this page recently moved into this tier and hasn't stabilized yet — a snapshot mid-transition looks worse than steady-state.

4. content_6aa54d6bbdbf6f24 — review_title_meta. Why: 50+ tier, essentially 0% CTR. Wrong if: the 50+ tier's own average (3.11%) is itself unreliable (n=315) — comparing against a shaky benchmark could misjudge this page.

5. content_7c6373141eae744a — review_title_meta. Why: 4-10 tier, CTR ~6x below tier average on 86,860 impressions. Wrong if: this is a duplicate/near-duplicate of another ranking page cannibalizing its own clicks — the fix would be consolidation, not a title rewrite.

6. content_b99ea6861864dea5 — review_title_meta. Why: 4-10 tier, CTR under half the tier average. Wrong if: seasonal/trending content past its peak — a title fix won't recover interest that's genuinely gone.

7. content_36e53e9c707674fc — review_title_meta. Why: 21-50 tier, CTR under 0.79% average with substantial volume (109,909 impressions). Wrong if: position itself is the real problem (rankings fluctuating), and CTR looks low simply because of where it's landing that day.

8. content_acbcc847f8996314 — review_title_meta. Why: 4-10 tier, CTR roughly 5x below average. Wrong if: this is a repeat appearance of a page we already reviewed in a prior cycle — re-flagging the same page without change history is wasted effort.

9. content_2de9a39d3482a269 — review_title_meta. Why: 50+ tier, near-zero CTR. Wrong if: same caveat as #4 — thin comparison group at this tier.

10. content_7172a7fad43f0998 — review_title_meta. Why: 4-10 tier, moderate gap but high volume (108,663 impressions) pushing it into the top 10 by raw score. Wrong if: the gap itself is small (0.0059) — this page is a volume-driven pick, not really a big underperformer relative to peers.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest picks:** #4 and #9 (both 50+ tier) lean on a tier average built from only 315 rows — a shaky benchmark. If that tier's "true" average CTR is actually lower than 3.11%, these two may not be real underperformers at all. #10 is weak for a different reason: it's mostly a volume-driven pick (high impressions) rather than a large relative gap — worth flagging as lower-confidence than the rest.

**Leakage check:** the score initially used full-month (days 1–31) CTR and position, which overlapped with the days 16–31 window the declining label is built from. I caught this, rebuilt the score using only days 1–15 (ctr_h1, position_tier_h1), and precision actually held steady or improved slightly (0.700/0.650/0.600 vs. the earlier leaky 0.600/0.550/0.460) — confirming the original result wasn't propped up by leakage. No product flags, future-window data, or label-derived columns are used as rule inputs; declining is used only to evaluate the rule afterward.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.